In [ ]:
import numpy as np
import glob
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from osgeo import gdal, osr, ogr
from shapely import from_wkt

import subprocess
import os
import re
import time
import scipy
import pandas as pd
import sys
from pandas import DataFrame
from IPython.display import Image
import sklearn
from sklearn import metrics
from matplotlib.pyplot import figure
from matplotlib.colors import ListedColormap
from ipywidgets import interactive
from rasterio.plot import show_hist
from rasterio.enums import ColorInterp
from rasterio.enums import Resampling
import datetime
from datetime import date, timedelta
from datetime import datetime, UTC
import math
os.environ['GDAL_DATA'] =  os.popen('gdal-config --datadir').read().rstrip() #resolves gdal directory value error

from pathlib import Path
import io
import rasterio
import zipfile
import h5py
from pyproj import CRS
import geopandas as gpd
import copy
import asf_search as asf
#import earthaccess
import json
from getpass import getpass
import requests

from IPython.display import display, HTML
from tqdm.notebook import tqdm
gdal.UseExceptions()

In [ ]:
work_dir=os.getcwd()
wkt_area='POLYGON((-100.063 19.849, -100.063 19.414, -99.723 19.414, \
-99.723 19.849, -100.063 19.849))'  #Atlacomulco
#print(work_dir)
dataset_name='NISAR'
product='GCOV'
polarizations='HH+HV'

EPSG_new='EPSG:4326'
target_x_res=100.0 #new resolution= 100 m
target_y_res=100.0 #new resolution= 100 m
os.chdir(os.path.join(work_dir,'tiles'))

In [ ]:
SAR_images_HH=glob.glob("*HHHH.tif")
SAR_images_HH=sorted(SAR_images_HH)
SAR_images_HV=glob.glob("*HVHV.tif")
SAR_images_HV=sorted(SAR_images_HV)
print(len(SAR_images_HH))
print(len(SAR_images_HV))

In [ ]:
arrs_HHHH=[]
arrs_HVHV=[]
for file_name in SAR_images_HH:
    ImageArray_HHHH=gdal.Open(file_name,gdal.GA_ReadOnly).ReadAsArray()
    ImageArray_HVHV=gdal.Open(file_name[0:-8]+'HVHV.tif',gdal.GA_ReadOnly).ReadAsArray()
    arrs_HHHH.append(ImageArray_HHHH)
    arrs_HVHV.append(ImageArray_HVHV)
    del ImageArray_HHHH, ImageArray_HVHV 

a_HHHH = np.array(arrs_HHHH,dtype=float)
a_HVHV = np.array(arrs_HVHV,dtype=float)
del arrs_HHHH, arrs_HVHV

print(np.shape(a_HHHH))

In [ ]:
ashape = np.shape(a_HHHH)
CPR = np.empty(ashape)
DpRVI = np.empty(ashape)
for ii in range(len(a_HHHH)):
    for jj in range(len(a_HHHH[ii])):
        for kk in range(len(a_HHHH[ii][jj])):
            CPR[ii][jj][kk] = a_HVHV[ii][jj][kk]/a_HHHH[ii][jj][kk]
            DpRVI[ii][jj][kk] = 4*a_HVHV[ii][jj][kk]/(a_HHHH[ii][jj][kk]+a_HVHV[ii][jj][kk])

In [ ]:
for ii in range(0,13,4):

    plt.figure()
    plt.rcParams['figure.figsize']=(20,10)
    plt.subplot(2,2,1)
    plt.imshow(10*np.log10(a_HHHH[ii][:][:]),vmin=-30,vmax=0,cmap='rainbow') #'Greys_r')
    plt.colorbar();
    plt.title(SAR_images_HH[ii][21:36])
    plt.subplot(2,2,2)
    plt.imshow(10*np.log10(a_HVHV[ii][:][:]),vmin=-30,vmax=0,cmap='rainbow') #'Greys_r')
    plt.colorbar();
    plt.title(SAR_images_HV[ii][21:36])
    plt.subplot(2,2,3)
    plt.imshow(10*np.log10(CPR[ii][:][:]),vmin=-15,vmax=0,cmap='rainbow') #'Greys_r')
    plt.colorbar();
    plt.title(SAR_images_HV[ii][21:31]+'_CPR')
    plt.subplot(2,2,4)
    plt.imshow(DpRVI[ii][:][:],vmin=0,vmax=1.5,cmap='rainbow') #'Greys_r')
    plt.colorbar();
    plt.title(SAR_images_HV[ii][21:31]+'_DpRVI')

    plt.savefig(SAR_images_HV[ii][21:31]+'.png', dpi=300, bbox_inches='tight') 
    plt.show()
    plt.close()

In [ ]:
# Creating a variable for the number of images (one for each date)
num_dates = len(SAR_images_HV)
print ("Number of dates:", num_dates)

# Get dimensions of the first SAR image--- should be the same for every image in the stack and the CDL
first_raster = gdal.Open(SAR_images_HV[0])
rows = first_raster.RasterYSize
cols = first_raster.RasterXSize
geotransform = first_raster.GetGeoTransform()
xres = geotransform[1]
yres = -geotransform[5]
xmin = geotransform[0]
ymax = geotransform[3]
spatialref = first_raster.GetProjectionRef()

In [ ]:
def statistical_parameters(a):
    
    # Calculate the mean for the time stack of images
    mean = np.nanmean(a, axis = 0)

    # Calculate the standard deviation for the time stack of images
    std = np.nanstd(a, axis = 0)

    # Calculate the coefficient of variation for the time stack of images 
    CV = std/mean
    return mean, std, CV

In [ ]:
mean_HVHV, std_HVHV, CV_HVHV = statistical_parameters(a_HVHV)
mean_HHHH, std_HHHH, CV_HHHH = statistical_parameters(a_HHHH)
mean_CPR, std_CPR, CV_CPR = statistical_parameters(CPR)
mean_DpRVI, std_DpRVI, CV_DpRVI = statistical_parameters(DpRVI)

In [ ]:
plt.figure() # Create figure
plt.rcParams['figure.figsize'] = (20,10)
plt.subplot(2,2,1)
plt.imshow(10*np.log10(mean_HVHV),vmin=-30,vmax=-7,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('Mean Radar-Cross Section HVHV (dB)')
plt.subplot(2,2,2)
plt.imshow(10*np.log10(mean_HHHH),vmin=-20,vmax=0,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('Mean Radar-Cross Section HHHH (dB)')
plt.subplot(2,2,3)
plt.imshow(10*np.log10(mean_CPR),vmin=-15,vmax=-5,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('Mean CPR (dB)')
plt.subplot(2,2,4)
plt.imshow(mean_DpRVI,vmin=0.2,vmax=1.1,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('Mean DpRVI (dB)')

plt.savefig('mean_plots.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure() # Create figure
plt.rcParams['figure.figsize'] = (20,10)
plt.subplot(2,2,1)
plt.imshow(10*np.log10(std_HVHV),vmin=-30,vmax=-15,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('STD Radar-Cross Section HVHV (dB)')
plt.subplot(2,2,2)
plt.imshow(10*np.log10(std_HHHH),vmin=-20,vmax=-5,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('STD Radar-Cross Section HHHH (dB)')
plt.subplot(2,2,3)
plt.imshow(10*np.log10(std_CPR),vmin=-20,vmax=-10,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('STD CPR (dB)')
plt.subplot(2,2,4)
plt.imshow(std_DpRVI,vmin=0.01,vmax=0.3,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('STD DpRVI (dB)')
plt.savefig('std_plots.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure() # Create figure
plt.rcParams['figure.figsize'] = (20,10)
plt.subplot(2,2,1)
plt.hist(np.clip(CV_HVHV, 0, 1.0).flatten(), bins=50) # Plot histogram
plt.title('Hist. of CV_HVHV values') # Set title
plt.xlabel('Value') # Add label
plt.ylabel('Frequency') # Add label

plt.subplot(2,2,2)
plt.hist(np.clip(CV_HHHH, 0, 1.0).flatten(), bins=50) # Plot histogram
plt.title('Hist. of CV_HHHH values') # Set title
plt.xlabel('Value') # Add label
plt.ylabel('Frequency') # Add label

plt.subplot(2,2,3)
plt.hist(np.clip(CV_CPR, 0, 1.0).flatten(), bins=50) # Plot histogram
plt.title('Hist. of CV_CPR values') # Set title
plt.xlabel('Value') # Add label
plt.ylabel('Frequency') # Add label

plt.subplot(2,2,4)
plt.hist(np.clip(CV_DpRVI, 0, 1.0).flatten(), bins=50) # Plot histogram
plt.title('Hist. of CV_DpRVI values') # Set title
plt.xlabel('Value') # Add label
plt.ylabel('Frequency') # Add label

plt.savefig('hist_CV_plots.png', dpi=300, bbox_inches='tight')
plt.show() # Display plot

In [ ]:
plt.figure() # Create figure
plt.rcParams['figure.figsize'] = (20,10)
plt.subplot(2,2,1)
plt.imshow(CV_HVHV,vmin=0.05,vmax=1,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('Coeff. of Var. HVHV');

plt.subplot(2,2,2)
plt.imshow(CV_HHHH,vmin=0.05,vmax=1,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('Coeff. of Var. HHHH');

plt.subplot(2,2,3)
plt.imshow(CV_CPR,vmin=0.05,vmax=1,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('Coeff. of Var. CPR');

plt.subplot(2,2,4)
plt.imshow(CV_CPR,vmin=0.05,vmax=1,interpolation='nearest',cmap='rainbow');
plt.colorbar();
plt.title('Coeff. of Var. DpRVI');

plt.savefig('CV_plots.png', dpi=300, bbox_inches='tight')
plt.show() # Display plot